1. ERA5 Inventory & Validation

In [ ]:
import xarray as xr

datacheck

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/100m_u_comp.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/100m_v_comp.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/2m_dewpoint_temp.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/2m_temp.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/10m_u_comp.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/10m_v_comp.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/boundary_lyr_h.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/ins_10m.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/m_sea_lvl_press.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/surface_press.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/surface_solar_rad.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/ttl_cloud_cvr.nc"
)

print(ds)

In [ ]:
ds = xr.open_dataset(
    "../data/raw/era5/2018/ttl_percipitation.nc"
)

print(ds)

2. Multiyear Loading

Summary

In [ ]:
from pathlib import Path
import pandas as pd

era5_root = Path("../data/raw/era5")

summary = []

for file in sorted(era5_root.glob("2018/*.nc")):

    ds = xr.open_dataset(file)

    var_name = list(ds.data_vars)[0]
    da = ds[var_name]

    summary.append({
        "File": file.name,
        "Variable": var_name,
        "Time Stamp": da.sizes["valid_time"],
        "Latitude Points": da.sizes["latitude"],
        "Longitude Points": da.sizes["longitude"],
        "Min": float(da.min().compute()),
        "Max": float(da.max().compute()),
        "Mean": float(da.mean().compute()),
        "Std": float(da.std().compute())
    })

summary_df = pd.DataFrame(summary)

summary_df

Combining years of variables

In [ ]:
from pathlib import Path
import xarray as xr

In [ ]:
variables = {
    "u100": "100m_u_comp.nc",
    "v100": "100m_v_comp.nc",
    "u10": "10m_u_comp.nc",
    "v10": "10m_v_comp.nc",
    "t2m": "2m_temp.nc",
    "d2m": "2m_dewpoint_temp.nc",
    "sp": "surface_press.nc",
    "msl": "m_sea_lvl_press.nc",
    "blh": "boundary_lyr_h.nc",
    "ssrd": "surface_solar_rad.nc",
    "tcc": "ttl_cloud_cvr.nc",
    "tp": "ttl_percipitation.nc",
    "i10fg": "ins_10m.nc"
}

era5 = {}

for name, filename in variables.items():
    files= sorted(
        Path("../data/raw/era5").glob(f"*/{filename}")
    )

    era5[name] = xr.open_mfdataset(
        files, combine = "by_coords"
    )

    print(f"{name} loaded")


In [ ]:
for name, ds in era5.items():
    var = list(ds.data_vars)[0]

    print(
        f"{name}:",
        ds.valid_time.min().values,
        ds.valid_time.max().values
    )

In [ ]:
datasets = [ds for ds in era5.values()]

era5_ds = xr.merge(datasets)

In [ ]:
print(list(era5_ds.data_vars))

3. Wind and Atmosphere Feature Engineering

In [ ]:
import numpy as np

100m Wind Speed

In [ ]:
era5_ds["ws100"] = np.sqrt(
    era5_ds["u100"]**2 +
    era5_ds["v100"]**2
)

10m Wind Speed

In [ ]:
era5_ds["ws10"] = np.sqrt(
    era5_ds["u10"]**2 +
    era5_ds["v10"]**2
)

100m Wind Direction

In [ ]:
era5_ds["wd100"] = (
    270 -
    np.degrees(
        np.arctan2(
            era5_ds["v100"],
            era5_ds["u100"]
        )
    )
) % 360

10m Wind Direction

In [ ]:
era5_ds["wd10"] = (
    270 -
    np.degrees(
        np.arctan2(
            era5_ds["u10"],
            era5_ds["v10"]
        )
    )
) % 360

Wind Shear to see how wind changes with height

In [ ]:
era5_ds["wind_shear"] = (
    era5_ds["ws100"] /
    era5_ds["ws10"]
)

Conversion of Relative Humidity from Kelvin to Celsius

In [ ]:
t = era5_ds["t2m"] - 273.15
td = era5_ds["d2m"] - 273.15

In [ ]:
era5_ds["relative_humidity"] = (
    100 *
    np.exp((17.625 * td)/(243.04 + td))
    /
    np.exp((17.625 * t)/ (243.04 + t))
)

Air Density for Wind Power

In [ ]:
era5_ds["air_density"] = (
    era5_ds["sp"]
    /
    (287.05 * era5_ds["t2m"])
)

In [ ]:
weather_features = era5_ds[
    [
        "ws100",
            "wd100",
            "ws10",
            "wd10",
            "wind_shear",
            "relative_humidity",
            "air_density",
            "blh",
            "tcc",
            "tp",
            "ssrd",
            "i10fg"
    ]
]

In [ ]:
era5_ds["ws100"].min().values
era5_ds["ws100"].max().values
era5_ds["ws100"].mean().values


In [ ]:
era5_ds["wd100"].min().values
era5_ds["wd100"].max().values

In [ ]:

era5_ds["relative_humidity"].min().values
era5_ds["relative_humidity"].max().values


In [ ]:
era5_ds["air_density"].mean().values

In [ ]:
weather_features.to_netcdf(
    "../data/interim/era5_processed/weather_feature.nc"
)

In [ ]:
%reset -f

In [ ]:
weather_features = xr.open_dataset(
    "../data/interim/era5_processed/weather_feature.nc"
)

In [ ]:
weather_features["valid_time"].head(10)

In [ ]:
weather_features.info()